# Tool Calling — học theo notebook B4

**Mục tiêu:** chạy tool bằng tay trước, rồi hiểu chuyện gì xảy ra khi model đề nghị gọi tool. Ví dụ tra một thiết bị trong kho phòng học. Học theo thứ tự từ trên xuống. Phần chính dùng `langchain-core` và dữ liệu mẫu. Cell OpenAI ở cuối mặc định tắt.

Cách viết theo `B4_ToolCalling_langchain/notebooks/01_chung_tu_va_chi_phi.ipynb`: hàm Python → `@tool` → `.invoke()`.

Lần đầu, chạy các bước 1–4 để hiểu một tool. Bước 5 dành cho lúc bạn tự thử OpenAI. Các bước 6–8 là bảng tra và bài tập offline cho BT2.

## 1. Viết hàm Python trước

`rooms` là dữ liệu giả của backend. Hàm `find_room` nhận mã và trả phòng tương ứng. `rooms.get(asset_id)` trả `None` nếu mã vắng trong bảng.

In [ ]:
rooms = {"KIT-7": "B2", "KIT-8": "C1"}


def find_room(asset_id: str) -> str | None:
    return rooms.get(asset_id)


print(find_room("KIT-7"))  # B2
print(find_room("KIT-0"))  # None

## 2. Bọc hàm bằng `@tool` rồi gọi bằng tay

`@tool` giúp LangChain lấy tên hàm, mô tả trong docstring và tham số từ khai báo kiểu. Tool dưới dùng lại `find_room`.

`lookup_room.invoke({"asset_id": "KIT-7"})` chạy tool ngay trong Python. Tên khóa `asset_id` khớp tên tham số trong hàm.

In [ ]:
from langchain_core.tools import tool


@tool
def lookup_room(asset_id: str) -> dict:
    """Tra phòng của một thiết bị theo mã thiết bị."""
    room = find_room(asset_id)
    if room is None:
        return {"error": "Không tìm thấy thiết bị"}
    return {"asset_id": asset_id, "room": room}


result = lookup_room.invoke({"asset_id": "KIT-7"})
print(result)
assert result["room"] == "B2"
assert lookup_room.invoke({"asset_id": "KIT-0"}) == {"error": "Không tìm thấy thiết bị"}

## 3. Dòng `bind_tools(...).invoke(...)` làm gì?

Mẫu trong B4 dùng hai thao tác nối nhau:

```python
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chosen_tool = lookup_room
first = llm.bind_tools([chosen_tool], tool_choice=chosen_tool.name).invoke(
    [HumanMessage(content=prompt)]
)
```

| Đoạn | Điều xảy ra |
|---|---|
| `ChatOpenAI(...)` | Tạo đối tượng để gọi model. |
| `llm.bind_tools([chosen_tool], ...)` | Gắn khai báo tool gồm tên, mô tả và tham số. `tool_choice=chosen_tool.name` yêu cầu model chọn đúng tool này trong ví dụ một tool. |
| `.invoke([HumanMessage(content=prompt)])` | Gửi câu hỏi cùng khai báo tool lên OpenAI. Model đọc chúng và trả phản hồi `first`. |
| `first.tool_calls[0]` | Lấy đề nghị gọi đầu tiên. Đề nghị chứa `name`, `args` và `id`. Ví dụ `args` là `{"asset_id": "KIT-7"}`. |

**Dừng ở đây thì mới có đề nghị.** Hàm Python thực sự chạy ở dòng `chosen_tool.invoke(call["args"])` trong bước sau. Bạn chưa cần đọc source nội bộ LangChain để theo dõi luồng này.

## 4. Thực thi đề nghị, rồi chuẩn bị trả kết quả cho model

Dưới đây là một `call` mẫu để tập offline. Backend kiểm tên có đúng tool được phép hay không rồi chạy tool. `ToolMessage` chứa kết quả dưới dạng JSON. `tool_call_id` lấy từ `call["id"]` để model biết kết quả này thuộc lời gọi nào.

In [ ]:
import json
from langchain_core.messages import ToolMessage

chosen_tool = lookup_room
call = {
    "name": "lookup_room",
    "args": {"asset_id": "KIT-7"},
    "id": "call_1",
}

if call["name"] != chosen_tool.name:
    raise ValueError("Tool không được phép")

result = chosen_tool.invoke(call["args"])
tool_reply = ToolMessage(
    content=json.dumps(result, ensure_ascii=False),
    tool_call_id=call["id"],
)

print(tool_reply.content)
print(tool_reply.tool_call_id)
assert tool_reply.tool_call_id == "call_1"

## 5. Tự thử với OpenAI khi sẵn sàng

Cell này gom đúng luồng của B4: gọi model lần đầu để lấy đề nghị → chạy tool → gọi model lần hai để viết câu trả lời. Lần hai nhận câu hỏi, phản hồi `first` có yêu cầu gọi tool và `ToolMessage` chứa kết quả.

Mặc định tắt. Khi tự bật, cần `langchain-openai`, `OPENAI_API_KEY` trong môi trường và quyền dùng model được chọn. Tên model lấy từ notebook B4.

In [ ]:
RUN_LLM_DEMO = False

if RUN_LLM_DEMO:
    from langchain_core.messages import HumanMessage
    from langchain_openai import ChatOpenAI

    prompt = "Thiết bị KIT-7 ở phòng nào?"
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    first = llm.bind_tools(
        [chosen_tool], tool_choice=chosen_tool.name
    ).invoke([HumanMessage(content=prompt)])

    print("Đề nghị:", first.tool_calls)

    if len(first.tool_calls) > 1:
        raise ValueError("Bước 5 chỉ minh họa một lời gọi. Xem bước 8.")

    if first.tool_calls:
        live_call = first.tool_calls[0]
        if live_call["name"] != chosen_tool.name:
            raise ValueError("Tool không được phép")

        live_result = chosen_tool.invoke(live_call["args"])
        print("Kết quả Python:", live_result)
        live_reply = ToolMessage(
            content=json.dumps(live_result, ensure_ascii=False),
            tool_call_id=live_call["id"],
        )
        final = llm.invoke([HumanMessage(content=prompt), first, live_reply])
        print("Trả lời:", final.content)
    else:
        print("Model chưa đề nghị gọi tool.")

**Tự thử:** ở bước 4, đổi mã thành `KIT-8`. Trước khi chạy, đoán nội dung `ToolMessage`. Tiếp theo, đổi `name` thành tên khác và xem dòng nào dừng chương trình.

Khi làm BT2, thay từ điển mẫu bằng dữ liệu backend. Học từng lớp khi cần: hàm thuật toán Python → tool → SQLite/SQLAlchemy → FastAPI và test → web chat. Mở [lộ trình stack B4–B7](../TOOL_CALLING_GUIDE.md) và [contract từng bài](../EXERCISE_CONTRACTS.md) khi đến lớp tương ứng.

## 6. Bảng tra cứu nhanh

### Các hàm và lệnh LangChain

| Lệnh | Đầu vào | Sau khi chạy, bạn có gì? |
|---|---|---|
| `ChatOpenAI(...)` | Tên model và cấu hình | Đối tượng `llm` để gọi OpenAI |
| `@tool` | Hàm Python có tên, docstring và kiểu tham số | Tool có thể gọi qua `.invoke()` |
| `SystemMessage(content=...)` | Hướng dẫn chung, ví dụ chỉ trả lời theo dữ liệu tool | Message đặt đầu danh sách nếu cần quy tắc cho model |
| `HumanMessage(content=...)` | Câu hỏi | Message của người dùng |
| `llm.bind_tools([chosen_tool])` | Danh sách tool | Model đã gắn khai báo tool. `tool_choice=chosen_tool.name` yêu cầu chọn tool cụ thể trong demo |
| `bound.invoke([question])` | Danh sách message | `AIMessage` có thể chứa đề nghị gọi tool |
| `chosen_tool.invoke(call["args"])` | Dict tham số | Kết quả chạy hàm Python. Ví dụ này trả dict |
| `ToolMessage(content=..., tool_call_id=...)` | Kết quả và ID lời gọi | Message để gửi kết quả tool cho model |
| `llm.invoke([question, first, tool_reply])` | Câu hỏi, đề nghị của model và kết quả tool | `AIMessage` chứa câu trả lời cuối trong ví dụ này |

Hai lệnh `.invoke()` trong luồng có vai trò khác nhau. Gọi trên **model** gửi yêu cầu lên OpenAI. Gọi trên **tool** chạy code Python của bạn.

### Những trường thường đọc

| Object | Trường | Ví dụ hoặc mục đích |
|---|---|---|
| Tool | `.name` | `"lookup_room"`. Dùng làm khóa trong danh sách tool được phép |
| Tool | `.description`, `.args` | Mô tả và schema tham số. Hữu ích khi muốn xem model được cung cấp thông tin gì |
| `HumanMessage` | `.content` | Câu hỏi người dùng gửi |
| `AIMessage` | `.tool_calls` | Danh sách lời gọi. Có thể có 0, 1 hoặc nhiều phần tử |
| Một phần tử của `.tool_calls` | `["name"]`, `["args"]`, `["id"]` | Tên tool, dict tham số và ID riêng của lần gọi |
| `ToolMessage` | `.content`, `.tool_call_id` | Kết quả đã đóng gói và ID khớp với `call["id"]` |
| `AIMessage` | `.content` | Nội dung trả lời. Ở lượt đề nghị tool, trường này có thể rỗng |
| `AIMessage` | `.invalid_tool_calls` | Xem khi LangChain không đọc được lời gọi, ví dụ tham số JSON sai cú pháp |
| `AIMessage` | `.usage_metadata` | Thống kê token nếu có |
| `AIMessage` | `.response_metadata` | Thông tin kèm theo từ nhà cung cấp, ví dụ tên model hoặc lý do kết thúc tùy tích hợp |

`first.tool_calls` là thuộc tính của message nên dùng dấu chấm. `call["args"]` là khóa trong một dict nên dùng ngoặc vuông. Khi tạo `ToolMessage`, lấy **ID của lời gọi** từ `call["id"]`. Một message có thể chứa nhiều lời gọi, mỗi lời gọi có ID riêng.

In [ ]:
print("Tên tool:", chosen_tool.name)
print("Mô tả:", chosen_tool.description)
print("Schema tham số:", chosen_tool.args)

print("Tên trong đề nghị:", call["name"])
print("Tham số model đề nghị:", call["args"])
print("ID trong đề nghị:", call["id"])
print("ID trong kết quả:", tool_reply.tool_call_id)

assert "asset_id" in chosen_tool.args
assert tool_reply.tool_call_id == call["id"]

## 7. Các hàm Python cần để đọc và đóng gói kết quả

| Lệnh | Khi nào dùng? |
|---|---|
| `dict.get(key)` | Tra mã thiết bị hoặc tìm tool theo tên. Trả `None` khi thiếu khóa nếu không chỉ định giá trị mặc định |
| `len(first.tool_calls)` | Đếm số lời gọi trước khi đọc phần tử `[0]` |
| `json.dumps(result, ensure_ascii=False)` | Chuyển dict thành chuỗi JSON để đưa vào `ToolMessage.content`. Giữ tiếng Việt dễ đọc |
| `json.loads(tool_reply.content)` | Đọc lại chuỗi JSON thành dict để kiểm kết quả hoặc hiển thị trace |
| `for call in calls` | Xử lý từng đề nghị nếu một lượt trả nhiều lời gọi |
| `messages.append(message)` | Thêm phản hồi model hoặc kết quả tool vào danh sách message theo đúng thứ tự |

Trong ví dụ này `json.loads()` đọc được vì chính bạn đã đóng gói bằng `json.dumps()`. `final.content` là câu trả lời thông thường, có thể đọc hoặc in trực tiếp.

In [ ]:
result_again = json.loads(tool_reply.content)

print(type(tool_reply.content))  # str
print(type(result_again))        # dict
print(result_again["room"])      # B2

assert result_again["room"] == "B2"

## 8. Khi một lượt có nhiều lời gọi tool

Ví dụ offline sau có hai đề nghị tra phòng. Cả hai dùng cùng tool và có ID khác nhau. `tools_by_name` là danh sách backend cho phép, biểu diễn bằng dict. Khi thêm tool thứ hai, bạn thêm tên và object của tool đó vào dict.

Mỗi lần lặp làm ba việc: tra tên tool, chạy với tham số tương ứng, tạo một `ToolMessage` có đúng ID. `trace` là bản ghi để bạn kiểm tra backend đã thực sự chạy gì.

In [ ]:
tools_by_name = {lookup_room.name: lookup_room}
calls = [
    {"name": "lookup_room", "args": {"asset_id": "KIT-7"}, "id": "call_a"},
    {"name": "lookup_room", "args": {"asset_id": "KIT-8"}, "id": "call_b"},
]

tool_replies = []
trace = []

for current_call in calls:
    selected_tool = tools_by_name.get(current_call["name"])
    if selected_tool is None:
        raise ValueError("Tool không được phép")

    result = selected_tool.invoke(current_call["args"])
    reply = ToolMessage(
        content=json.dumps(result, ensure_ascii=False),
        tool_call_id=current_call["id"],
    )
    tool_replies.append(reply)
    trace.append({
        "name": current_call["name"],
        "args": current_call["args"],
        "id": current_call["id"],
        "result": result,
    })

print(trace)

assert tool_replies[0].tool_call_id == "call_a"
assert tool_replies[1].tool_call_id == "call_b"
assert trace[0]["result"]["room"] == "B2"
assert trace[1]["result"]["room"] == "C1"

### Đưa đoạn trên vào vòng lặp BT2

Ở chế độ thật, `calls` lấy từ `first.tool_calls`. Bạn giữ `question` và `first` trong danh sách message, rồi thêm từng `ToolMessage` vào sau bằng `append()`. Sau khi có kết quả cho cả lượt, gọi model lại.

Trong BT2, nếu model cần lấy ID từ tool đầu rồi mới gọi tool tiếp theo, bạn lặp lại các bước trên qua nhiều lượt. Đặt giới hạn số lượt ngay từ đầu và dừng khi model đã trả lời hoặc có lỗi cần báo. Tool có trách nhiệm đọc dữ liệu backend và kiểm nghiệp vụ. Schema đúng kiểu vẫn có thể chứa mã thiết bị không tồn tại.

**Tự thử:** đổi mã của `call_b` thành `KIT-0`. Trước khi chạy lại cell, dự đoán ID nào giữ nguyên và kết quả nào đổi thành lỗi. Sau khi quan sát trace, khôi phục mã để các assert mẫu đạt lại.

Tài liệu đối chiếu: [Luồng Tool Calling](https://docs.langchain.com/oss/python/langchain/models#tool-calling) · [Các object message](https://reference.langchain.com/python/langchain-core/messages).